# Live Scoreboards: Kafka -> Spark -> BigQuery
Reads flight events from Kafka, joins `airlines.csv` and `airports.csv` from GCS (`degroup5_storage`), computes daily running counts, and upserts into `deproject1-512109.live_scoreboard`.

In [ ]:
# Install Google Cloud BigQuery client library
!pip install google-cloud-bigquery

In [ ]:
# ================================================================ SECTION 0: settings + Spark session
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import (col, from_json, to_timestamp, window, count, sum as spark_sum,
                                   when, round as round_, broadcast, upper, trim, current_timestamp)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# ---- 0.1 Settings: everything that depends on YOUR account / setup lives here
PROJECT_ID = "deproject1-512109"
BUCKET = "degroup5_storage"            # the bucket that holds airlines.csv and airports.csv
BQ_DATASET = "live_scoreboard"         # must exist, in the same region as the bucket

AIRLINES_PATH = f"gs://{BUCKET}/airlines.csv"
AIRPORTS_PATH = f"gs://{BUCKET}/airports.csv"

TABLE_AIRLINE = f"{PROJECT_ID}.{BQ_DATASET}.live_airline_scoreboard"
TABLE_AIRPORT = f"{PROJECT_ID}.{BQ_DATASET}.live_airport_scoreboard"

KAFKA_SERVERS = "kafka:29092"          # INTERNAL listener (Spark runs inside the docker network)
TOPIC = "flights"
CHECKPOINT_DIR = "/home/jovyan/checkpoint"
LATE_MINUTES = 15                      # late = left 15+ minutes after schedule
WATERMARK = "3 hours"                  # how long after midnight stragglers are still accepted
TRIGGER = "1 minute"                   # how often the scoreboard is updated

# ---- 0.2 Spark session
sparkConf = SparkConf()
sparkConf.setMaster("spark://spark-master:7077")
sparkConf.setAppName("flights_scoreboard")
sparkConf.set("spark.driver.memory", "2g")
sparkConf.set("spark.cores.max", "2")                # 2 workers x 1 core; stop other notebooks first (two queries run)
sparkConf.set("spark.executor.cores", "1")
sparkConf.set("spark.driver.cores", "1")
sparkConf.set("spark.sql.shuffle.partitions", "4")   # default 200 is far too many for a tiny cluster
sparkConf.set("spark.sql.session.timeZone", "UTC")   # so "1 day" windows start at midnight of the flight date

spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()

hconf = spark.sparkContext._jsc.hadoopConfiguration()          # how to read gs:// paths (same as Lab9_3)
hconf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hconf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
spark.conf.set("temporaryGcsBucket", BUCKET)                   # BigQuery connector's temp files

In [ ]:
# ================================================================ SECTION 1: read from Kafka + parse
flightSchema = StructType([
    StructField("airline_code", StringType(), True),
    StructField("flight_no", StringType(), True),
    StructField("origin", StringType(), True),
    StructField("dest", StringType(), True),
    StructField("event_time", StringType(), True),   # scheduled departure
    StructField("dep_delay", DoubleType(), True),    # null when cancelled
    StructField("cancelled", IntegerType(), True),
])

raw = (spark.readStream.format("kafka")
       .option("kafka.bootstrap.servers", KAFKA_SERVERS)
       .option("subscribe", TOPIC)
       .option("startingOffsets", "earliest")
       .option("failOnDataLoss", "false")
       .load())

flights = (raw.select(from_json(col("value").cast("string"), flightSchema).alias("f"))
           .select("f.*")
           .withColumn("event_time", to_timestamp("event_time"))
           # tidy codes so " ord" and "ORD" match in the join
           .withColumn("airline_code", upper(trim(col("airline_code"))))
           .withColumn("origin", upper(trim(col("origin"))))
           # business rules: cancelled flights are not "late"
           .withColumn("is_cancelled", when(col("cancelled") == 1, 1).otherwise(0))
           .withColumn("is_late", when((col("cancelled") == 0) & (col("dep_delay") >= LATE_MINUTES), 1).otherwise(0)))

In [ ]:
# ================================================================ SECTION 2: read + clean the lookup tables (GCS)
airlines = (spark.read.option("header", True).csv(AIRLINES_PATH)
            .select(upper(trim(col("airline_code"))).alias("airline_code"),
                    trim(col("airline")).alias("airline_name"))
            .dropDuplicates(["airline_code"])
            .cache())

airports = (spark.read.option("header", True).csv(AIRPORTS_PATH)
            .select(upper(trim(col("iata_code"))).alias("origin"),       # joined to the DEPARTURE airport
                    trim(col("airport_name")).alias("airport_name"),
                    trim(col("city")).alias("city"),
                    trim(col("state")).alias("state"))
            .dropDuplicates(["origin"])
            .cache())

# cache() + count() = read from GCS once now, not again every minute
print("airlines:", airlines.count(), "rows | airports:", airports.count(), "rows")

In [ ]:
# ================================================================ SECTION 3: stream-static joins
enriched = (flights
            .join(broadcast(airlines), "airline_code", "left")   # left: never silently drop a flight
            .join(broadcast(airports), "origin", "left")
            .fillna({"airline_name": "Unknown", "airport_name": "Unknown",
                     "city": "Unknown", "state": "Unknown"}))
# Data-quality check: rows with name 'Unknown' show up in the scoreboards (see the SQL check in the guide).

In [ ]:
# ================================================================ SECTION 4: watermark + 1-day window + running counts
def scoreboard(df, keys):
    return (df.withWatermark("event_time", WATERMARK)
            .groupBy(window(col("event_time"), "1 day").alias("day"), *keys)
            .agg(count("*").alias("flights"),
                 spark_sum("is_late").alias("late_flights"),
                 spark_sum("is_cancelled").alias("cancelled_flights"))
            .withColumn("on_time_pct",
                        round_((col("flights") - col("late_flights") - col("cancelled_flights"))
                               / col("flights") * 100, 1))
            .select(col("day.start").cast("date").alias("flight_date"), *keys,
                    "flights", "late_flights", "cancelled_flights", "on_time_pct"))


airline_board = scoreboard(enriched, ["airline_code", "airline_name"])
airport_board = (scoreboard(enriched, ["origin", "airport_name", "city", "state"])
                 .withColumnRenamed("origin", "airport_code"))

In [ ]:
# ================================================================ SECTION 5: write to BigQuery (upsert with MERGE)
from google.cloud import bigquery      # needs: pip install google-cloud-bigquery (see Dockerfile_1 note)
bq = bigquery.Client(project=PROJECT_ID)


def make_writer(board, target, keys, create_sql):
    """Returns a foreachBatch function: changed rows -> staging table -> MERGE into target."""
    staging = target + "_staging"
    bq.query(create_sql).result()                       # create the target table if it does not exist
    cols = board.columns + ["updated_at"]
    on = " AND ".join(f"T.{k} = S.{k}" for k in keys)
    sets = ", ".join(f"{c} = S.{c}" for c in cols if c not in keys)
    merge_sql = (f"MERGE `{target}` T USING `{staging}` S ON {on} "
                 f"WHEN MATCHED THEN UPDATE SET {sets} "
                 f"WHEN NOT MATCHED THEN INSERT ({', '.join(cols)}) "
                 f"VALUES ({', '.join('S.' + c for c in cols)})")

    def write(batch_df, batch_id):
        # Update mode: batch_df holds ONLY the rows that changed in this micro-batch.
        if batch_df.isEmpty():
            return
        (batch_df.withColumn("updated_at", current_timestamp())
         .write.format("bigquery").option("table", staging)
         .mode("overwrite")                             # staging only ever holds the latest micro-batch
         .save())
        bq.query(merge_sql).result()
    return write


write_airline = make_writer(airline_board, TABLE_AIRLINE, ["flight_date", "airline_code"], f"""
CREATE TABLE IF NOT EXISTS `{TABLE_AIRLINE}` (
  flight_date DATE, airline_code STRING, airline_name STRING,
  flights INT64, late_flights INT64, cancelled_flights INT64,
  on_time_pct FLOAT64, updated_at TIMESTAMP)""")

write_airport = make_writer(airport_board, TABLE_AIRPORT, ["flight_date", "airport_code"], f"""
CREATE TABLE IF NOT EXISTS `{TABLE_AIRPORT}` (
  flight_date DATE, airport_code STRING, airport_name STRING, city STRING, state STRING,
  flights INT64, late_flights INT64, cancelled_flights INT64,
  on_time_pct FLOAT64, updated_at TIMESTAMP)""")

In [ ]:
# ================================================================ SECTION 6: trigger + checkpoint + start
def start(board, writer, name):
    return (board.writeStream.queryName(name)
            .outputMode("update")
            .trigger(processingTime=TRIGGER)
            .option("checkpointLocation", f"{CHECKPOINT_DIR}/{name}")   # one checkpoint folder per query
            .foreachBatch(writer)
            .start())


q_airline = start(airline_board, write_airline, "airline_scoreboard")
q_airport = start(airport_board, write_airport, "airport_scoreboard")

try:
    spark.streams.awaitAnyTermination()
except KeyboardInterrupt:
    for q in spark.streams.active:
        q.stop()
    airlines.unpersist()
    airports.unpersist()
    spark.stop()
    print("Stopped the streaming queries and the spark context")